# ALL CODES FOR MY FINAL YEAR PROJECT

# Pima and Kidney Basic SMOTE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io
import requests
import warnings
import random

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ========================================================
# 1. SMOTE-MRS IMPLEMENTATION
# ========================================================
def custom_resample(X, y, k_neighbors=3):
    X = np.array(X)
    y = np.array(y)
    counts = pd.Series(y).value_counts()
    minority_class = counts.idxmin()
    X_min = X[y == minority_class]
    n_min = len(X_min)
    
    if n_min > 1:
        nn = NearestNeighbors(n_neighbors=min(n_min, k_neighbors + 1)).fit(X_min)
        dist, idxs = nn.kneighbors(X_min)
        
        synthetic = []
        for _ in range(counts.max() - n_min):
            i = random.randint(0, n_min - 1)
            neighbor = random.choice(idxs[i][1:])
            diff = X_min[neighbor] - X_min[i]
            synthetic.append(X_min[i] + random.random() * diff)
        
        if synthetic:
            X = np.vstack([X, np.array(synthetic)])
            y = np.append(y, [minority_class] * len(synthetic))
    
    return X, y

class SMOTEMRS_Base:
    def __init__(self, R=5):
        self.R = R
        
    def fit_resample(self, X, y):
        kmeans = KMeans(n_clusters=self.R, random_state=42, n_init='auto')
        clusters = kmeans.fit_predict(X)
        
        X_res, y_res = [], []
        for r in range(self.R):
            mask = (clusters == r)
            if not any(mask): continue
            
            X_sub, y_sub = X[mask], y[mask]
            if len(np.unique(y_sub)) > 1:
                X_s, y_s = custom_resample(X_sub, y_sub)
                X_res.append(X_s)
                y_res.append(y_s)
            else:
                X_res.append(X_sub)
                y_res.append(y_sub)
                
        return np.vstack(X_res), np.concatenate(y_res)

# ========================================================
# 2. DATASET LOADERS
# ========================================================
def load_diabetes():
    url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
    cols = ['Preg', 'Glu', 'BP', 'Skin', 'Ins', 'BMI', 'Ped', 'Age', 'Outcome']
    df = pd.read_csv(io.StringIO(requests.get(url).content.decode('utf-8')), names=cols)
    X = StandardScaler().fit_transform(df.drop('Outcome', axis=1))
    y = df['Outcome'].values
    return X, y, "Pima Diabetes"
def load_ckd():
    # Load from Kaggle input folder
    df = pd.read_csv('/datasets/kidney_disease.csv')
    
    # Drop the 'id' column
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
    
    # Replace '?' with NaN
    df = df.replace('?', np.nan)
    
    # Clean the target column (remove whitespace/tabs)
    df['classification'] = df['classification'].str.strip()
    
    # Convert target to binary: ckd=1, notckd=0
    df['classification'] = df['classification'].map({'ckd': 1, 'notckd': 0})
    
    # Drop any rows where target is still NaN (bad data)
    df = df.dropna(subset=['classification'])
    
    # Fill missing values for features
    for col in df.columns:
        if col == 'classification':
            continue
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)
    
    # Encode remaining categorical columns
    le = LabelEncoder()
    for col in df.select_dtypes(include='object').columns:
        df[col] = le.fit_transform(df[col].astype(str))
    
    X = StandardScaler().fit_transform(df.drop('classification', axis=1))
    y = df['classification'].astype(int).values
    return X, y, "UCI CKD (Kaggle)"


# ========================================================
# 3. EVALUATION FUNCTION
# ========================================================
def evaluate(X, y, dataset_name):
    models = {
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Naïve Bayes': GaussianNB(),
        'SVM': SVC(probability=True, random_state=42)
    }
    
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    for name, model in models.items():
        m = {'acc': [], 'rec': [], 'f1': [], 'auc': []}
        for train_idx, test_idx in skf.split(X, y):
            X_tr, y_tr = X[train_idx], y[train_idx]
            X_te, y_te = X[test_idx], y[test_idx]
            
            engine = SMOTEMRS_Base(R=5)
            X_res, y_res = engine.fit_resample(X_tr, y_tr)
            
            model.fit(X_res, y_res)
            preds = model.predict(X_te)
            probs = model.predict_proba(X_te)[:, 1]
            
            m['acc'].append(accuracy_score(y_te, preds))
            m['rec'].append(recall_score(y_te, preds))
            m['f1'].append(f1_score(y_te, preds))
            m['auc'].append(roc_auc_score(y_te, probs))
            
        results.append({
            'Dataset': dataset_name,
            'Model': name,
            'Accuracy': round(np.mean(m['acc']), 4),
            'Recall': round(np.mean(m['rec']), 4),
            'F1': round(np.mean(m['f1']), 4),
            'AUC': round(np.mean(m['auc']), 4)
        })
        print(f"✅ {dataset_name} - {name} Done")
        
    return results

# ========================================================
# 4. RUN ON BOTH DATASETS
# ========================================================
print("🚀 SMOTE-MRS Base Paper Replication\n")

all_results = []

# Dataset 1: Pima Diabetes
X1, y1, name1 = load_diabetes()
print(f"📊 {name1}: {len(y1)} samples, Class distribution: {np.bincount(y1)}")
all_results.extend(evaluate(X1, y1, name1))

# Dataset 2: UCI CKD (from Kaggle)
X2, y2, name2 = load_ckd()
print(f"\n📊 {name2}: {len(y2)} samples, Class distribution: {np.bincount(y2)}")
all_results.extend(evaluate(X2, y2, name2))

# Final Results Table
results_df = pd.DataFrame(all_results)
print("\n" + "="*70)
print("           BASE PAPER REPLICATION - BOTH DATASETS")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

# Adaptive Pima and CKD

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io
import requests
import warnings
import random

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ========================================================
# 1. CUSTOM SMOTE IMPLEMENTATION (Same as Base)
# ========================================================
def custom_resample(X, y, k_neighbors=3):
    X = np.array(X)
    y = np.array(y)
    counts = pd.Series(y).value_counts()
    minority_class = counts.idxmin()
    X_min = X[y == minority_class]
    n_min = len(X_min)
    
    if n_min > 1:
        nn = NearestNeighbors(n_neighbors=min(n_min, k_neighbors + 1)).fit(X_min)
        dist, idxs = nn.kneighbors(X_min)
        
        synthetic = []
        for _ in range(counts.max() - n_min):
            i = random.randint(0, n_min - 1)
            neighbor = random.choice(idxs[i][1:])
            diff = X_min[neighbor] - X_min[i]
            synthetic.append(X_min[i] + random.random() * diff)
        
        if synthetic:
            X = np.vstack([X, np.array(synthetic)])
            y = np.append(y, [minority_class] * len(synthetic))
    
    return X, y

# ========================================================
# 2. ADAPTIVE SMOTE-MRS IMPLEMENTATION ⭐ YOUR INNOVATION
# ========================================================
class AdaptiveSMOTEMRS:
    """
    Adaptive SMOTE-MRS: Applies SMOTE only to clusters with high Imbalance Ratio (IR)
    
    Innovation:
    - Calculates IR for each cluster
    - Only applies SMOTE if IR >= threshold (default 1.5)
    - Skips SMOTE for already-balanced clusters
    - Result: Better quality synthetic data, no noise amplification
    """
    def __init__(self, R=5, ir_threshold=1.5):
        self.R = R
        self.ir_threshold = ir_threshold
        
    def fit_resample(self, X, y):
        kmeans = KMeans(n_clusters=self.R, random_state=42, n_init='auto')
        clusters = kmeans.fit_predict(X)
        
        X_res, y_res = [], []
        
        for r in range(self.R):
            mask = (clusters == r)
            if not any(mask): 
                continue
            
            X_sub, y_sub = X[mask], y[mask]
            unique_classes = np.unique(y_sub)
            
            # If cluster is pure (only one class), keep original
            if len(unique_classes) < 2:
                X_res.append(X_sub)
                y_res.append(y_sub)
                continue
            
            # Calculate Imbalance Ratio for this cluster
            counts = pd.Series(y_sub).value_counts()
            majority_count = counts.max()
            minority_count = counts.min()
            ir = majority_count / minority_count
            
            # ADAPTIVE DECISION: Only apply SMOTE if IR >= threshold
            if ir >= self.ir_threshold:
                # High imbalance → Apply SMOTE
                X_s, y_s = custom_resample(X_sub, y_sub)
                X_res.append(X_s)
                y_res.append(y_s)
            else:
                # Low imbalance → Skip SMOTE (avoid noise)
                X_res.append(X_sub)
                y_res.append(y_sub)
                
        return np.vstack(X_res), np.concatenate(y_res)

# ========================================================
# 3. DATASET LOADERS (Same as Base)
# ========================================================
def load_diabetes():
    url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
    cols = ['Preg', 'Glu', 'BP', 'Skin', 'Ins', 'BMI', 'Ped', 'Age', 'Outcome']
    df = pd.read_csv(io.StringIO(requests.get(url).content.decode('utf-8')), names=cols)
    X = StandardScaler().fit_transform(df.drop('Outcome', axis=1))
    y = df['Outcome'].values
    return X, y, "Pima Diabetes"

def load_ckd():
    # Load from Kaggle input folder
    df = pd.read_csv('/datasets/kidney_disease.csv')
    
    # Drop the 'id' column
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
    
    # Replace '?' with NaN
    df = df.replace('?', np.nan)
    
    # Clean the target column (remove whitespace/tabs)
    df['classification'] = df['classification'].str.strip()
    
    # Convert target to binary: ckd=1, notckd=0
    df['classification'] = df['classification'].map({'ckd': 1, 'notckd': 0})
    
    # Drop any rows where target is still NaN (bad data)
    df = df.dropna(subset=['classification'])
    
    # Fill missing values for features
    for col in df.columns:
        if col == 'classification':
            continue
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)
    
    # Encode remaining categorical columns
    le = LabelEncoder()
    for col in df.select_dtypes(include='object').columns:
        df[col] = le.fit_transform(df[col].astype(str))
    
    X = StandardScaler().fit_transform(df.drop('classification', axis=1))
    y = df['classification'].astype(int).values
    return X, y, "UCI CKD (Kaggle)"


# ========================================================
# 4. EVALUATION FUNCTION (Adaptive)
# ========================================================
def evaluate_adaptive(X, y, dataset_name):
    models = {
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Naïve Bayes': GaussianNB(),
        'SVM': SVC(probability=True, random_state=42)
    }
    
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    for name, model in models.items():
        m = {'acc': [], 'rec': [], 'f1': [], 'auc': []}
        for train_idx, test_idx in skf.split(X, y):
            X_tr, y_tr = X[train_idx], y[train_idx]
            X_te, y_te = X[test_idx], y[test_idx]
            
            # ADAPTIVE SMOTE-MRS (IR threshold = 1.5)
            engine = AdaptiveSMOTEMRS(R=5, ir_threshold=1.5)
            X_res, y_res = engine.fit_resample(X_tr, y_tr)
            
            model.fit(X_res, y_res)
            preds = model.predict(X_te)
            probs = model.predict_proba(X_te)[:, 1]
            
            m['acc'].append(accuracy_score(y_te, preds))
            m['rec'].append(recall_score(y_te, preds))
            m['f1'].append(f1_score(y_te, preds))
            m['auc'].append(roc_auc_score(y_te, probs))
            
        results.append({
            'Dataset': dataset_name,
            'Model': name,
            'Accuracy': round(np.mean(m['acc']), 4),
            'Recall': round(np.mean(m['rec']), 4),
            'F1': round(np.mean(m['f1']), 4),
            'AUC': round(np.mean(m['auc']), 4)
        })
        print(f"✅ {dataset_name} - {name} Done")
        
    return results

# ========================================================
# 5. RUN ON BOTH DATASETS
# ========================================================
print("🚀 ADAPTIVE SMOTE-MRS Implementation\n")

all_results = []

# Dataset 1: Pima Diabetes
X1, y1, name1 = load_diabetes()
print(f"📊 {name1}: {len(y1)} samples, Class distribution: {np.bincount(y1)}")
all_results.extend(evaluate_adaptive(X1, y1, name1))

# Dataset 2: UCI CKD (from Kaggle)
X2, y2, name2 = load_ckd()
print(f"\n📊 {name2}: {len(y2)} samples, Class distribution: {np.bincount(y2)}")
all_results.extend(evaluate_adaptive(X2, y2, name2))

# Final Results Table
results_df = pd.DataFrame(all_results)
print("\n" + "="*70)
print("        ADAPTIVE SMOTE-MRS - BOTH DATASETS (IR THRESHOLD = 1.5)")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)


# Stroke Basic SMOTE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression  # Fast alternative to SVM
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ========================================================
# 1. SMOTE-MRS IMPLEMENTATION
# ========================================================
def custom_resample(X, y, k_neighbors=3):
    X = np.array(X)
    y = np.array(y)
    counts = pd.Series(y).value_counts()
    minority_class = counts.idxmin()
    X_min = X[y == minority_class]
    n_min = len(X_min)
    
    if n_min > 1:
        nn = NearestNeighbors(n_neighbors=min(n_min, k_neighbors + 1)).fit(X_min)
        dist, idxs = nn.kneighbors(X_min)
        
        synthetic = []
        for _ in range(counts.max() - n_min):
            i = random.randint(0, n_min - 1)
            neighbor = random.choice(idxs[i][1:])
            diff = X_min[neighbor] - X_min[i]
            synthetic.append(X_min[i] + random.random() * diff)
        
        if synthetic:
            X = np.vstack([X, np.array(synthetic)])
            y = np.append(y, [minority_class] * len(synthetic))
    
    return X, y

class SMOTEMRS_Base:
    def __init__(self, R=5):
        self.R = R
        
    def fit_resample(self, X, y):
        kmeans = KMeans(n_clusters=self.R, random_state=42, n_init='auto')
        clusters = kmeans.fit_predict(X)
        
        X_res, y_res = [], []
        for r in range(self.R):
            mask = (clusters == r)
            if not any(mask): continue
            
            X_sub, y_sub = X[mask], y[mask]
            if len(np.unique(y_sub)) > 1:
                X_s, y_s = custom_resample(X_sub, y_sub)
                X_res.append(X_s)
                y_res.append(y_s)
            else:
                X_res.append(X_sub)
                y_res.append(y_sub)
                
        return np.vstack(X_res), np.concatenate(y_res)

# ========================================================
# 2. LOAD STROKE DATASET
# ========================================================
def load_stroke():
    df = pd.read_csv('/datasets/healthcare-dataset-stroke-data.csv')
    df = df.drop('id', axis=1)
    df['bmi'].fillna(df['bmi'].median(), inplace=True)
    
    le = LabelEncoder()
    for col in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']:
        df[col] = le.fit_transform(df[col].astype(str))
    
    X = StandardScaler().fit_transform(df.drop('stroke', axis=1))
    y = df['stroke'].values
    return X, y

# ========================================================
# 3. FAST EVALUATION (No SVM)
# ========================================================
def evaluate(X, y):
    # Using fast models only
    models = {
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'Naïve Bayes': GaussianNB(),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
    }
    
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    for name, model in models.items():
        m = {'acc': [], 'rec': [], 'f1': [], 'auc': []}
        for train_idx, test_idx in skf.split(X, y):
            X_tr, y_tr = X[train_idx], y[train_idx]
            X_te, y_te = X[test_idx], y[test_idx]
            
            engine = SMOTEMRS_Base(R=5)
            X_res, y_res = engine.fit_resample(X_tr, y_tr)
            
            model.fit(X_res, y_res)
            preds = model.predict(X_te)
            probs = model.predict_proba(X_te)[:, 1]
            
            m['acc'].append(accuracy_score(y_te, preds))
            m['rec'].append(recall_score(y_te, preds))
            m['f1'].append(f1_score(y_te, preds))
            m['auc'].append(roc_auc_score(y_te, probs))
            
        results.append({
            'Model': name,
            'Accuracy': round(np.mean(m['acc']), 4),
            'Recall': round(np.mean(m['rec']), 4),
            'F1': round(np.mean(m['f1']), 4),
            'AUC': round(np.mean(m['auc']), 4)
        })
        print(f"✅ {name} Done")
        
    return pd.DataFrame(results)

# ========================================================
# 4. RUN
# ========================================================
print("🚀 SMOTE-MRS Base Paper - Stroke Prediction Dataset\n")

X, y = load_stroke()
print(f"📊 Samples: {len(y)}")
print(f"📊 Class distribution: {np.bincount(y)}")
print(f"📊 Imbalance Ratio: {np.bincount(y)[0] / np.bincount(y)[1]:.2f}:1\n")

results = evaluate(X, y)

print("\n" + "="*60)
print("      BASE PAPER RESULTS - STROKE PREDICTION")
print("="*60)
print(results.to_string(index=False))
print("="*60)




# STROKE Adaptive SMOTE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression  # Fast alternative to SVM
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ========================================================
# 1. CUSTOM SMOTE IMPLEMENTATION (Same as Base)
# ========================================================
def custom_resample(X, y, k_neighbors=3):
    X = np.array(X)
    y = np.array(y)
    counts = pd.Series(y).value_counts()
    minority_class = counts.idxmin()
    X_min = X[y == minority_class]
    n_min = len(X_min)
    
    if n_min > 1:
        nn = NearestNeighbors(n_neighbors=min(n_min, k_neighbors + 1)).fit(X_min)
        dist, idxs = nn.kneighbors(X_min)
        
        synthetic = []
        for _ in range(counts.max() - n_min):
            i = random.randint(0, n_min - 1)
            neighbor = random.choice(idxs[i][1:])
            diff = X_min[neighbor] - X_min[i]
            synthetic.append(X_min[i] + random.random() * diff)
        
        if synthetic:
            X = np.vstack([X, np.array(synthetic)])
            y = np.append(y, [minority_class] * len(synthetic))
    
    return X, y

# ========================================================
# 2. ADAPTIVE SMOTE-MRS IMPLEMENTATION ⭐ YOUR INNOVATION
# ========================================================
class AdaptiveSMOTEMRS:
    """
    Adaptive SMOTE-MRS: Applies SMOTE only to clusters with high Imbalance Ratio (IR)
    
    Innovation:
    - Calculates IR for each cluster
    - Only applies SMOTE if IR >= threshold (default 1.5)
    - Skips SMOTE for already-balanced clusters
    - Result: Better quality synthetic data, no noise amplification
    """
    def __init__(self, R=5, ir_threshold=1.5):
        self.R = R
        self.ir_threshold = ir_threshold
        
    def fit_resample(self, X, y):
        kmeans = KMeans(n_clusters=self.R, random_state=42, n_init='auto')
        clusters = kmeans.fit_predict(X)
        
        X_res, y_res = [], []
        
        for r in range(self.R):
            mask = (clusters == r)
            if not any(mask): 
                continue
            
            X_sub, y_sub = X[mask], y[mask]
            unique_classes = np.unique(y_sub)
            
            # If cluster is pure (only one class), keep original
            if len(unique_classes) < 2:
                X_res.append(X_sub)
                y_res.append(y_sub)
                continue
            
            # Calculate Imbalance Ratio for this cluster
            counts = pd.Series(y_sub).value_counts()
            majority_count = counts.max()
            minority_count = counts.min()
            ir = majority_count / minority_count
            
            # ADAPTIVE DECISION: Only apply SMOTE if IR >= threshold
            if ir >= self.ir_threshold:
                # High imbalance → Apply SMOTE
                X_s, y_s = custom_resample(X_sub, y_sub)
                X_res.append(X_s)
                y_res.append(y_s)
            else:
                # Low imbalance → Skip SMOTE (avoid noise)
                X_res.append(X_sub)
                y_res.append(y_sub)
                
        return np.vstack(X_res), np.concatenate(y_res)

# ========================================================
# 3. LOAD STROKE DATASET (Same as Base)
# ========================================================
def load_stroke():
    df = pd.read_csv('/kaggle/input/datasets/fedesoriano/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv')
    df = df.drop('id', axis=1)
    df['bmi'].fillna(df['bmi'].median(), inplace=True)
    
    le = LabelEncoder()
    for col in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']:
        df[col] = le.fit_transform(df[col].astype(str))
    
    X = StandardScaler().fit_transform(df.drop('stroke', axis=1))
    y = df['stroke'].values
    return X, y

# ========================================================
# 4. ADAPTIVE EVALUATION
# ========================================================
def evaluate_adaptive(X, y):
    # Using fast models only
    models = {
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'Naïve Bayes': GaussianNB(),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
    }
    
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    for name, model in models.items():
        m = {'acc': [], 'rec': [], 'f1': [], 'auc': []}
        for train_idx, test_idx in skf.split(X, y):
            X_tr, y_tr = X[train_idx], y[train_idx]
            X_te, y_te = X[test_idx], y[test_idx]
            
            # ADAPTIVE SMOTE-MRS (IR threshold = 1.5)
            engine = AdaptiveSMOTEMRS(R=5, ir_threshold=1.5)
            X_res, y_res = engine.fit_resample(X_tr, y_tr)
            
            model.fit(X_res, y_res)
            preds = model.predict(X_te)
            probs = model.predict_proba(X_te)[:, 1]
            
            m['acc'].append(accuracy_score(y_te, preds))
            m['rec'].append(recall_score(y_te, preds))
            m['f1'].append(f1_score(y_te, preds))
            m['auc'].append(roc_auc_score(y_te, probs))
            
        results.append({
            'Model': name,
            'Accuracy': round(np.mean(m['acc']), 4),
            'Recall': round(np.mean(m['rec']), 4),
            'F1': round(np.mean(m['f1']), 4),
            'AUC': round(np.mean(m['auc']), 4)
        })
        print(f"✅ {name} Done")
        
    return pd.DataFrame(results)

# ========================================================
# 5. RUN
# ========================================================
print("🚀 ADAPTIVE SMOTE-MRS - Stroke Prediction Dataset\n")

X, y = load_stroke()
print(f"📊 Samples: {len(y)}")
print(f"📊 Class distribution: {np.bincount(y)}")
print(f"📊 Imbalance Ratio: {np.bincount(y)[0] / np.bincount(y)[1]:.2f}:1\n")

results = evaluate_adaptive(X, y)

print("\n" + "="*60)
print("   ADAPTIVE SMOTE-MRS RESULTS - STROKE PREDICTION")
print("           (IR THRESHOLD = 1.5)")
print("="*60)
print(results.to_string(index=False))
print("="*60)


# Implmenting XGBoost

# XGBoost pima ckd Base

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io
import requests
import warnings
import random

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ========================================================
# 1. CUSTOM SMOTE IMPLEMENTATION
# ========================================================
def custom_resample(X, y, k_neighbors=3):
    X = np.array(X)
    y = np.array(y)
    counts = pd.Series(y).value_counts()
    minority_class = counts.idxmin()
    X_min = X[y == minority_class]
    n_min = len(X_min)
    
    if n_min > 1:
        nn = NearestNeighbors(n_neighbors=min(n_min, k_neighbors + 1)).fit(X_min)
        dist, idxs = nn.kneighbors(X_min)
        
        synthetic = []
        for _ in range(counts.max() - n_min):
            i = random.randint(0, n_min - 1)
            neighbor = random.choice(idxs[i][1:])
            diff = X_min[neighbor] - X_min[i]
            synthetic.append(X_min[i] + random.random() * diff)
        
        if synthetic:
            X = np.vstack([X, np.array(synthetic)])
            y = np.append(y, [minority_class] * len(synthetic))
    
    return X, y

# ========================================================
# 2. BASE SMOTE-MRS IMPLEMENTATION
# ========================================================
class SMOTEMRS_Base:
    def __init__(self, R=5):
        self.R = R
        
    def fit_resample(self, X, y):
        kmeans = KMeans(n_clusters=self.R, random_state=42, n_init='auto')
        clusters = kmeans.fit_predict(X)
        
        X_res, y_res = [], []
        for r in range(self.R):
            mask = (clusters == r)
            if not any(mask): continue
            
            X_sub, y_sub = X[mask], y[mask]
            if len(np.unique(y_sub)) > 1:
                X_s, y_s = custom_resample(X_sub, y_sub)
                X_res.append(X_s)
                y_res.append(y_s)
            else:
                X_res.append(X_sub)
                y_res.append(y_sub)
                
        return np.vstack(X_res), np.concatenate(y_res)

# ========================================================
# 3. DATASET LOADERS
# ========================================================
def load_diabetes():
    url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
    cols = ['Preg', 'Glu', 'BP', 'Skin', 'Ins', 'BMI', 'Ped', 'Age', 'Outcome']
    df = pd.read_csv(io.StringIO(requests.get(url).content.decode('utf-8')), names=cols)
    X = StandardScaler().fit_transform(df.drop('Outcome', axis=1))
    y = df['Outcome'].values
    return X, y, "Pima Diabetes"

def load_ckd():
    df = pd.read_csv('/kaggle/input/datasets/mansoordaku/ckdisease/kidney_disease.csv')
    
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
    
    df = df.replace('?', np.nan)
    df['classification'] = df['classification'].str.strip()
    df['classification'] = df['classification'].map({'ckd': 1, 'notckd': 0})
    df = df.dropna(subset=['classification'])
    
    for col in df.columns:
        if col == 'classification':
            continue
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)
    
    le = LabelEncoder()
    for col in df.select_dtypes(include='object').columns:
        df[col] = le.fit_transform(df[col].astype(str))
    
    X = StandardScaler().fit_transform(df.drop('classification', axis=1))
    y = df['classification'].astype(int).values
    return X, y, "UCI CKD (Kaggle)"

# ========================================================
# 4. XGBOOST EVALUATION
# ========================================================
def evaluate_xgboost(X, y, dataset_name):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    # Calculate scale_pos_weight for imbalanced data
    counts = pd.Series(y).value_counts()
    minority_count = counts.min()
    majority_count = counts.max()
    scale_pos_weight = majority_count / minority_count

    print(f"\n📊 {dataset_name} - Scale Pos Weight: {scale_pos_weight:.2f}")
    
    m = {'acc': [], 'rec': [], 'f1': [], 'auc': []}
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[test_idx], y[test_idx]
        
        # Apply Base SMOTE-MRS
        engine = SMOTEMRS_Base(R=5)
        X_res, y_res = engine.fit_resample(X_tr, y_tr)
        
        # Train XGBoost
        xgb = XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            tree_method='hist',
            n_jobs=-1,
            verbose=0
        )
        xgb.fit(X_res, y_res)
        
        preds = xgb.predict(X_te)
        probs = xgb.predict_proba(X_te)[:, 1]
        
        m['acc'].append(accuracy_score(y_te, preds))
        m['rec'].append(recall_score(y_te, preds))
        m['f1'].append(f1_score(y_te, preds))
        m['auc'].append(roc_auc_score(y_te, probs))
        
        print(f"  Fold {fold}: Recall={m['rec'][-1]:.4f}, Accuracy={m['acc'][-1]:.4f}")
    
    results.append({
        'Dataset': dataset_name,
        'Model': 'XGBoost',
        'Accuracy': round(np.mean(m['acc']), 4),
        'Recall': round(np.mean(m['rec']), 4),
        'F1': round(np.mean(m['f1']), 4),
        'AUC': round(np.mean(m['auc']), 4)
    })
    print(f"✅ {dataset_name} - XGBoost Done")
    
    return results

# ========================================================
# 5. RUN ON BOTH DATASETS
# ========================================================
print("🚀 XGBoost with Base SMOTE-MRS - Pima & CKD\n")

all_results = []

# Dataset 1: Pima Diabetes
X1, y1, name1 = load_diabetes()
print(f"📊 {name1}: {len(y1)} samples, Class distribution: {np.bincount(y1)}")
all_results.extend(evaluate_xgboost(X1, y1, name1))

# Dataset 2: UCI CKD
X2, y2, name2 = load_ckd()
print(f"\n📊 {name2}: {len(y2)} samples, Class distribution: {np.bincount(y2)}")
all_results.extend(evaluate_xgboost(X2, y2, name2))

# Final Results Table
results_df = pd.DataFrame(all_results)
print("\n" + "="*70)
print("        XGBoost with BASE SMOTE-MRS - BOTH DATASETS")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

# pip install xgboost lightgbm

# Adaptive XGBoost ckd pima

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io
import requests
import warnings
import random

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ========================================================
# 1. CUSTOM SMOTE IMPLEMENTATION
# ========================================================
def custom_resample(X, y, k_neighbors=3):
    X = np.array(X)
    y = np.array(y)
    counts = pd.Series(y).value_counts()
    minority_class = counts.idxmin()
    X_min = X[y == minority_class]
    n_min = len(X_min)
    
    if n_min > 1:
        nn = NearestNeighbors(n_neighbors=min(n_min, k_neighbors + 1)).fit(X_min)
        dist, idxs = nn.kneighbors(X_min)
        
        synthetic = []
        for _ in range(counts.max() - n_min):
            i = random.randint(0, n_min - 1)
            neighbor = random.choice(idxs[i][1:])
            diff = X_min[neighbor] - X_min[i]
            synthetic.append(X_min[i] + random.random() * diff)
        
        if synthetic:
            X = np.vstack([X, np.array(synthetic)])
            y = np.append(y, [minority_class] * len(synthetic))
    
    return X, y

# ========================================================
# 2. ADAPTIVE SMOTE-MRS IMPLEMENTATION ⭐
# ========================================================
class AdaptiveSMOTEMRS:
    def __init__(self, R=5, ir_threshold=1.5):
        self.R = R
        self.ir_threshold = ir_threshold
        
    def fit_resample(self, X, y):
        kmeans = KMeans(n_clusters=self.R, random_state=42, n_init='auto')
        clusters = kmeans.fit_predict(X)
        
        X_res, y_res = [], []
        
        for r in range(self.R):
            mask = (clusters == r)
            if not any(mask): 
                continue
            
            X_sub, y_sub = X[mask], y[mask]
            unique_classes = np.unique(y_sub)
            
            if len(unique_classes) < 2:
                X_res.append(X_sub)
                y_res.append(y_sub)
                continue
            
            counts = pd.Series(y_sub).value_counts()
            majority_count = counts.max()
            minority_count = counts.min()
            ir = majority_count / minority_count
            
            if ir >= self.ir_threshold:
                X_s, y_s = custom_resample(X_sub, y_sub)
                X_res.append(X_s)
                y_res.append(y_s)
            else:
                X_res.append(X_sub)
                y_res.append(y_sub)
                
        return np.vstack(X_res), np.concatenate(y_res)

# ========================================================
# 3. DATASET LOADERS
# ========================================================
def load_diabetes():
    url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
    cols = ['Preg', 'Glu', 'BP', 'Skin', 'Ins', 'BMI', 'Ped', 'Age', 'Outcome']
    df = pd.read_csv(io.StringIO(requests.get(url).content.decode('utf-8')), names=cols)
    X = StandardScaler().fit_transform(df.drop('Outcome', axis=1))
    y = df['Outcome'].values
    return X, y, "Pima Diabetes"

def load_ckd():
    df = pd.read_csv('/kaggle/input/datasets/mansoordaku/ckdisease/kidney_disease.csv')
    
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
    
    df = df.replace('?', np.nan)
    df['classification'] = df['classification'].str.strip()
    df['classification'] = df['classification'].map({'ckd': 1, 'notckd': 0})
    df = df.dropna(subset=['classification'])
    
    for col in df.columns:
        if col == 'classification':
            continue
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)
    
    le = LabelEncoder()
    for col in df.select_dtypes(include='object').columns:
        df[col] = le.fit_transform(df[col].astype(str))
    
    X = StandardScaler().fit_transform(df.drop('classification', axis=1))
    y = df['classification'].astype(int).values
    return X, y, "UCI CKD (Kaggle)"

# ========================================================
# 4. XGBOOST EVALUATION WITH ADAPTIVE
# ========================================================
def evaluate_xgboost_adaptive(X, y, dataset_name):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    # Calculate scale_pos_weight
    counts = pd.Series(y).value_counts()
    minority_count = counts.min()
    majority_count = counts.max()
    scale_pos_weight = majority_count / minority_count

    print(f"\n📊 {dataset_name} - Scale Pos Weight: {scale_pos_weight:.2f}")
    
    m = {'acc': [], 'rec': [], 'f1': [], 'auc': []}
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[test_idx], y[test_idx]
        
        # Apply ADAPTIVE SMOTE-MRS
        engine = AdaptiveSMOTEMRS(R=5, ir_threshold=1.5)
        X_res, y_res = engine.fit_resample(X_tr, y_tr)
        
        # Train XGBoost
        xgb = XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            tree_method='hist',
            n_jobs=-1,
            verbose=0
        )
        xgb.fit(X_res, y_res)
        
        preds = xgb.predict(X_te)
        probs = xgb.predict_proba(X_te)[:, 1]
        
        m['acc'].append(accuracy_score(y_te, preds))
        m['rec'].append(recall_score(y_te, preds))
        m['f1'].append(f1_score(y_te, preds))
        m['auc'].append(roc_auc_score(y_te, probs))
        
        print(f"  Fold {fold}: Recall={m['rec'][-1]:.4f}, Accuracy={m['acc'][-1]:.4f}")
    
    results.append({
        'Dataset': dataset_name,
        'Model': 'XGBoost',
        'Accuracy': round(np.mean(m['acc']), 4),
        'Recall': round(np.mean(m['rec']), 4),
        'F1': round(np.mean(m['f1']), 4),
        'AUC': round(np.mean(m['auc']), 4)
    })
    print(f"✅ {dataset_name} - XGBoost (Adaptive) Done")
    
    return results

# ========================================================
# 5. RUN ON BOTH DATASETS
# ========================================================
print("🚀 XGBoost with ADAPTIVE SMOTE-MRS - Pima & CKD\n")

all_results = []

# Dataset 1: Pima Diabetes
X1, y1, name1 = load_diabetes()
print(f"📊 {name1}: {len(y1)} samples, Class distribution: {np.bincount(y1)}")
all_results.extend(evaluate_xgboost_adaptive(X1, y1, name1))

# Dataset 2: UCI CKD
X2, y2, name2 = load_ckd()
print(f"\n📊 {name2}: {len(y2)} samples, Class distribution: {np.bincount(y2)}")
all_results.extend(evaluate_xgboost_adaptive(X2, y2, name2))

# Final Results Table
results_df = pd.DataFrame(all_results)
print("\n" + "="*70)
print("     XGBoost with ADAPTIVE SMOTE-MRS - BOTH DATASETS")
print("         (IR THRESHOLD = 1.5)")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)


# sTROKE bASE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ========================================================
# 1. CUSTOM SMOTE IMPLEMENTATION
# ========================================================
def custom_resample(X, y, k_neighbors=3):
    X = np.array(X)
    y = np.array(y)
    counts = pd.Series(y).value_counts()
    minority_class = counts.idxmin()
    X_min = X[y == minority_class]
    n_min = len(X_min)
    
    if n_min > 1:
        nn = NearestNeighbors(n_neighbors=min(n_min, k_neighbors + 1)).fit(X_min)
        dist, idxs = nn.kneighbors(X_min)
        
        synthetic = []
        for _ in range(counts.max() - n_min):
            i = random.randint(0, n_min - 1)
            neighbor = random.choice(idxs[i][1:])
            diff = X_min[neighbor] - X_min[i]
            synthetic.append(X_min[i] + random.random() * diff)
        
        if synthetic:
            X = np.vstack([X, np.array(synthetic)])
            y = np.append(y, [minority_class] * len(synthetic))
    
    return X, y

# ========================================================
# 2. BASE SMOTE-MRS IMPLEMENTATION
# ========================================================
class SMOTEMRS_Base:
    def __init__(self, R=5):
        self.R = R
        
    def fit_resample(self, X, y):
        kmeans = KMeans(n_clusters=self.R, random_state=42, n_init='auto')
        clusters = kmeans.fit_predict(X)
        
        X_res, y_res = [], []
        for r in range(self.R):
            mask = (clusters == r)
            if not any(mask): continue
            
            X_sub, y_sub = X[mask], y[mask]
            if len(np.unique(y_sub)) > 1:
                X_s, y_s = custom_resample(X_sub, y_sub)
                X_res.append(X_s)
                y_res.append(y_s)
            else:
                X_res.append(X_sub)
                y_res.append(y_sub)
                
        return np.vstack(X_res), np.concatenate(y_res)

# ========================================================
# 3. LOAD STROKE DATASET
# ========================================================
def load_stroke():
    df = pd.read_csv('/kaggle/input/datasets/fedesoriano/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv')
    df = df.drop('id', axis=1)
    df['bmi'].fillna(df['bmi'].median(), inplace=True)
    
    le = LabelEncoder()
    for col in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']:
        df[col] = le.fit_transform(df[col].astype(str))
    
    X = StandardScaler().fit_transform(df.drop('stroke', axis=1))
    y = df['stroke'].values
    return X, y

# ========================================================
# 4. XGBOOST EVALUATION (WITHOUT scale_pos_weight)
# ========================================================
def evaluate_xgboost(X, y):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    print(f"\n📊 Stroke Prediction Dataset")
    
    m = {'acc': [], 'rec': [], 'f1': [], 'auc': []}
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[test_idx], y[test_idx]
        
        # Apply Base SMOTE-MRS
        engine = SMOTEMRS_Base(R=5)
        X_res, y_res = engine.fit_resample(X_tr, y_tr)
        
        # Train XGBoost WITHOUT scale_pos_weight
        xgb = XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            tree_method='hist',
            n_jobs=-1,
            verbose=0
        )
        xgb.fit(X_res, y_res)
        
        preds = xgb.predict(X_te)
        probs = xgb.predict_proba(X_te)[:, 1]
        
        m['acc'].append(accuracy_score(y_te, preds))
        m['rec'].append(recall_score(y_te, preds))
        m['f1'].append(f1_score(y_te, preds))
        m['auc'].append(roc_auc_score(y_te, probs))
        
        print(f"  Fold {fold}: Recall={m['rec'][-1]:.4f}, Accuracy={m['acc'][-1]:.4f}")
    
    results.append({
        'Model': 'XGBoost',
        'Accuracy': round(np.mean(m['acc']), 4),
        'Recall': round(np.mean(m['rec']), 4),
        'F1': round(np.mean(m['f1']), 4),
        'AUC': round(np.mean(m['auc']), 4)
    })
    print(f"✅ Stroke Prediction - XGBoost (Base) Done")
    
    return pd.DataFrame(results)

# ========================================================
# 5. RUN
# ========================================================
print("🚀 XGBoost with Base SMOTE-MRS - Stroke Prediction Dataset\n")

X, y = load_stroke()
print(f"📊 Samples: {len(y)}")
print(f"📊 Class distribution: {np.bincount(y)}")
print(f"📊 Imbalance Ratio: {np.bincount(y)[0] / np.bincount(y)[1]:.2f}:1\n")

results = evaluate_xgboost(X, y)

print("\n" + "="*60)
print("   XGBoost with BASE SMOTE-MRS - STROKE PREDICTION")
print("="*60)
print(results.to_string(index=False))
print("="*60)


# sTROKE aDAPTIVE SMOTE-MRS

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ========================================================
# 1. CUSTOM SMOTE IMPLEMENTATION
# ========================================================
def custom_resample(X, y, k_neighbors=3):
    X = np.array(X)
    y = np.array(y)
    counts = pd.Series(y).value_counts()
    minority_class = counts.idxmin()
    X_min = X[y == minority_class]
    n_min = len(X_min)
    
    if n_min > 1:
        nn = NearestNeighbors(n_neighbors=min(n_min, k_neighbors + 1)).fit(X_min)
        dist, idxs = nn.kneighbors(X_min)
        
        synthetic = []
        for _ in range(counts.max() - n_min):
            i = random.randint(0, n_min - 1)
            neighbor = random.choice(idxs[i][1:])
            diff = X_min[neighbor] - X_min[i]
            synthetic.append(X_min[i] + random.random() * diff)
        
        if synthetic:
            X = np.vstack([X, np.array(synthetic)])
            y = np.append(y, [minority_class] * len(synthetic))
    
    return X, y

# ========================================================
# 2. ADAPTIVE SMOTE-MRS IMPLEMENTATION ⭐
# ========================================================
class AdaptiveSMOTEMRS:
    def __init__(self, R=5, ir_threshold=1.5):
        self.R = R
        self.ir_threshold = ir_threshold
        
    def fit_resample(self, X, y):
        kmeans = KMeans(n_clusters=self.R, random_state=42, n_init='auto')
        clusters = kmeans.fit_predict(X)
        
        X_res, y_res = [], []
        
        for r in range(self.R):
            mask = (clusters == r)
            if not any(mask): 
                continue
            
            X_sub, y_sub = X[mask], y[mask]
            unique_classes = np.unique(y_sub)
            
            if len(unique_classes) < 2:
                X_res.append(X_sub)
                y_res.append(y_sub)
                continue
            
            counts = pd.Series(y_sub).value_counts()
            majority_count = counts.max()
            minority_count = counts.min()
            ir = majority_count / minority_count
            
            if ir >= self.ir_threshold:
                X_s, y_s = custom_resample(X_sub, y_sub)
                X_res.append(X_s)
                y_res.append(y_s)
            else:
                X_res.append(X_sub)
                y_res.append(y_sub)
                
        return np.vstack(X_res), np.concatenate(y_res)

# ========================================================
# 3. LOAD STROKE DATASET
# ========================================================
def load_stroke():
    df = pd.read_csv('/kaggle/input/datasets/fedesoriano/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv')
    df = df.drop('id', axis=1)
    df['bmi'].fillna(df['bmi'].median(), inplace=True)
    
    le = LabelEncoder()
    for col in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']:
        df[col] = le.fit_transform(df[col].astype(str))
    
    X = StandardScaler().fit_transform(df.drop('stroke', axis=1))
    y = df['stroke'].values
    return X, y

# ========================================================
# 4. XGBOOST EVALUATION WITH ADAPTIVE (WITHOUT scale_pos_weight)
# ========================================================
def evaluate_xgboost_adaptive(X, y):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    print(f"\n📊 Stroke Prediction Dataset")
    
    m = {'acc': [], 'rec': [], 'f1': [], 'auc': []}
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[test_idx], y[test_idx]
        
        # Apply ADAPTIVE SMOTE-MRS
        engine = AdaptiveSMOTEMRS(R=5, ir_threshold=1.5)
        X_res, y_res = engine.fit_resample(X_tr, y_tr)
        
        # Train XGBoost WITHOUT scale_pos_weight
        xgb = XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            tree_method='hist',
            n_jobs=-1,
            verbose=0
        )
        xgb.fit(X_res, y_res)
        
        preds = xgb.predict(X_te)
        probs = xgb.predict_proba(X_te)[:, 1]
        
        m['acc'].append(accuracy_score(y_te, preds))
        m['rec'].append(recall_score(y_te, preds))
        m['f1'].append(f1_score(y_te, preds))
        m['auc'].append(roc_auc_score(y_te, probs))
        
        print(f"  Fold {fold}: Recall={m['rec'][-1]:.4f}, Accuracy={m['acc'][-1]:.4f}")
    
    results.append({
        'Model': 'XGBoost',
        'Accuracy': round(np.mean(m['acc']), 4),
        'Recall': round(np.mean(m['rec']), 4),
        'F1': round(np.mean(m['f1']), 4),
        'AUC': round(np.mean(m['auc']), 4)
    })
    print(f"✅ Stroke Prediction - XGBoost (Adaptive) Done")
    
    return pd.DataFrame(results)

# ========================================================
# 5. RUN
# ========================================================
print("🚀 XGBoost with ADAPTIVE SMOTE-MRS - Stroke Prediction Dataset\n")

X, y = load_stroke()
print(f"📊 Samples: {len(y)}")
print(f"📊 Class distribution: {np.bincount(y)}")
print(f"📊 Imbalance Ratio: {np.bincount(y)[0] / np.bincount(y)[1]:.2f}:1\n")

results = evaluate_xgboost_adaptive(X, y)

print("\n" + "="*60)
print(" XGBoost with ADAPTIVE SMOTE-MRS - STROKE PREDICTION")
print("     (IR THRESHOLD = 1.5)")
print("="*60)
print(results.to_string(index=False))
print("="*60)


# Implmenting Light GBM 

# Base Pima ckd

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io
import requests
import warnings
import random

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ========================================================
# 1. CUSTOM SMOTE IMPLEMENTATION
# ========================================================
def custom_resample(X, y, k_neighbors=3):
    X = np.array(X)
    y = np.array(y)
    counts = pd.Series(y).value_counts()
    minority_class = counts.idxmin()
    X_min = X[y == minority_class]
    n_min = len(X_min)
    
    if n_min > 1:
        nn = NearestNeighbors(n_neighbors=min(n_min, k_neighbors + 1)).fit(X_min)
        dist, idxs = nn.kneighbors(X_min)
        
        synthetic = []
        for _ in range(counts.max() - n_min):
            i = random.randint(0, n_min - 1)
            neighbor = random.choice(idxs[i][1:])
            diff = X_min[neighbor] - X_min[i]
            synthetic.append(X_min[i] + random.random() * diff)
        
        if synthetic:
            X = np.vstack([X, np.array(synthetic)])
            y = np.append(y, [minority_class] * len(synthetic))
    
    return X, y

# ========================================================
# 2. BASE SMOTE-MRS IMPLEMENTATION
# ========================================================
class SMOTEMRS_Base:
    def __init__(self, R=5):
        self.R = R
        
    def fit_resample(self, X, y):
        kmeans = KMeans(n_clusters=self.R, random_state=42, n_init='auto')
        clusters = kmeans.fit_predict(X)
        
        X_res, y_res = [], []
        for r in range(self.R):
            mask = (clusters == r)
            if not any(mask): continue
            
            X_sub, y_sub = X[mask], y[mask]
            if len(np.unique(y_sub)) > 1:
                X_s, y_s = custom_resample(X_sub, y_sub)
                X_res.append(X_s)
                y_res.append(y_s)
            else:
                X_res.append(X_sub)
                y_res.append(y_sub)
                
        return np.vstack(X_res), np.concatenate(y_res)

# ========================================================
# 3. DATASET LOADERS
# ========================================================
def load_diabetes():
    url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
    cols = ['Preg', 'Glu', 'BP', 'Skin', 'Ins', 'BMI', 'Ped', 'Age', 'Outcome']
    df = pd.read_csv(io.StringIO(requests.get(url).content.decode('utf-8')), names=cols)
    X = StandardScaler().fit_transform(df.drop('Outcome', axis=1))
    y = df['Outcome'].values
    return X, y, "Pima Diabetes"

def load_ckd():
    df = pd.read_csv('/kaggle/input/datasets/mansoordaku/ckdisease/kidney_disease.csv')
    
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
    
    df = df.replace('?', np.nan)
    df['classification'] = df['classification'].str.strip()
    df['classification'] = df['classification'].map({'ckd': 1, 'notckd': 0})
    df = df.dropna(subset=['classification'])
    
    for col in df.columns:
        if col == 'classification':
            continue
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)
    
    le = LabelEncoder()
    for col in df.select_dtypes(include='object').columns:
        df[col] = le.fit_transform(df[col].astype(str))
    
    X = StandardScaler().fit_transform(df.drop('classification', axis=1))
    y = df['classification'].astype(int).values
    return X, y, "UCI CKD (Kaggle)"

# ========================================================
# 4. LIGHTGBM EVALUATION
# ========================================================
def evaluate_lightgbm(X, y, dataset_name):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    # Calculate scale_pos_weight
    counts = pd.Series(y).value_counts()
    minority_count = counts.min()
    majority_count = counts.max()
    scale_pos_weight = majority_count / minority_count

    print(f"\n📊 {dataset_name} - Scale Pos Weight: {scale_pos_weight:.2f}")
    
    m = {'acc': [], 'rec': [], 'f1': [], 'auc': []}
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[test_idx], y[test_idx]
        
        # Apply Base SMOTE-MRS
        engine = SMOTEMRS_Base(R=5)
        X_res, y_res = engine.fit_resample(X_tr, y_tr)
        
        # Train LightGBM
        lgb = LGBMClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            is_unbalance=True,  # Handle imbalance
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
        lgb.fit(X_res, y_res)
        
        preds = lgb.predict(X_te)
        probs = lgb.predict_proba(X_te)[:, 1]
        
        m['acc'].append(accuracy_score(y_te, preds))
        m['rec'].append(recall_score(y_te, preds))
        m['f1'].append(f1_score(y_te, preds))
        m['auc'].append(roc_auc_score(y_te, probs))
        
        print(f"  Fold {fold}: Recall={m['rec'][-1]:.4f}, Accuracy={m['acc'][-1]:.4f}")
    
    results.append({
        'Dataset': dataset_name,
        'Model': 'LightGBM',
        'Accuracy': round(np.mean(m['acc']), 4),
        'Recall': round(np.mean(m['rec']), 4),
        'F1': round(np.mean(m['f1']), 4),
        'AUC': round(np.mean(m['auc']), 4)
    })
    print(f"✅ {dataset_name} - LightGBM Done")
    
    return results

# ========================================================
# 5. RUN ON BOTH DATASETS
# ========================================================
print("🚀 LightGBM with Base SMOTE-MRS - Pima & CKD\n")

all_results = []

# Dataset 1: Pima Diabetes
X1, y1, name1 = load_diabetes()
print(f"📊 {name1}: {len(y1)} samples, Class distribution: {np.bincount(y1)}")
all_results.extend(evaluate_lightgbm(X1, y1, name1))

# Dataset 2: UCI CKD
X2, y2, name2 = load_ckd()
print(f"\n📊 {name2}: {len(y2)} samples, Class distribution: {np.bincount(y2)}")
all_results.extend(evaluate_lightgbm(X2, y2, name2))

# Final Results Table
results_df = pd.DataFrame(all_results)
print("\n" + "="*70)
print("       LightGBM with BASE SMOTE-MRS - BOTH DATASETS")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)


# Adaptive light gbm pima ckd

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io
import requests
import warnings
import random

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ========================================================
# 1. CUSTOM SMOTE IMPLEMENTATION
# ========================================================
def custom_resample(X, y, k_neighbors=3):
    X = np.array(X)
    y = np.array(y)
    counts = pd.Series(y).value_counts()
    minority_class = counts.idxmin()
    X_min = X[y == minority_class]
    n_min = len(X_min)
    
    if n_min > 1:
        nn = NearestNeighbors(n_neighbors=min(n_min, k_neighbors + 1)).fit(X_min)
        dist, idxs = nn.kneighbors(X_min)
        
        synthetic = []
        for _ in range(counts.max() - n_min):
            i = random.randint(0, n_min - 1)
            neighbor = random.choice(idxs[i][1:])
            diff = X_min[neighbor] - X_min[i]
            synthetic.append(X_min[i] + random.random() * diff)
        
        if synthetic:
            X = np.vstack([X, np.array(synthetic)])
            y = np.append(y, [minority_class] * len(synthetic))
    
    return X, y

# ========================================================
# 2. ADAPTIVE SMOTE-MRS IMPLEMENTATION ⭐
# ========================================================
class AdaptiveSMOTEMRS:
    def __init__(self, R=5, ir_threshold=1.5):
        self.R = R
        self.ir_threshold = ir_threshold
        
    def fit_resample(self, X, y):
        kmeans = KMeans(n_clusters=self.R, random_state=42, n_init='auto')
        clusters = kmeans.fit_predict(X)
        
        X_res, y_res = [], []
        
        for r in range(self.R):
            mask = (clusters == r)
            if not any(mask): 
                continue
            
            X_sub, y_sub = X[mask], y[mask]
            unique_classes = np.unique(y_sub)
            
            if len(unique_classes) < 2:
                X_res.append(X_sub)
                y_res.append(y_sub)
                continue
            
            counts = pd.Series(y_sub).value_counts()
            majority_count = counts.max()
            minority_count = counts.min()
            ir = majority_count / minority_count
            
            if ir >= self.ir_threshold:
                X_s, y_s = custom_resample(X_sub, y_sub)
                X_res.append(X_s)
                y_res.append(y_s)
            else:
                X_res.append(X_sub)
                y_res.append(y_sub)
                
        return np.vstack(X_res), np.concatenate(y_res)

# ========================================================
# 3. DATASET LOADERS
# ========================================================
def load_diabetes():
    url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
    cols = ['Preg', 'Glu', 'BP', 'Skin', 'Ins', 'BMI', 'Ped', 'Age', 'Outcome']
    df = pd.read_csv(io.StringIO(requests.get(url).content.decode('utf-8')), names=cols)
    X = StandardScaler().fit_transform(df.drop('Outcome', axis=1))
    y = df['Outcome'].values
    return X, y, "Pima Diabetes"

def load_ckd():
    df = pd.read_csv('/kaggle/input/datasets/mansoordaku/ckdisease/kidney_disease.csv')
    
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
    
    df = df.replace('?', np.nan)
    df['classification'] = df['classification'].str.strip()
    df['classification'] = df['classification'].map({'ckd': 1, 'notckd': 0})
    df = df.dropna(subset=['classification'])
    
    for col in df.columns:
        if col == 'classification':
            continue
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)
    
    le = LabelEncoder()
    for col in df.select_dtypes(include='object').columns:
        df[col] = le.fit_transform(df[col].astype(str))
    
    X = StandardScaler().fit_transform(df.drop('classification', axis=1))
    y = df['classification'].astype(int).values
    return X, y, "UCI CKD (Kaggle)"

# ========================================================
# 4. LIGHTGBM EVALUATION WITH ADAPTIVE
# ========================================================
def evaluate_lightgbm_adaptive(X, y, dataset_name):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    print(f"\n📊 {dataset_name}")
    
    m = {'acc': [], 'rec': [], 'f1': [], 'auc': []}
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[test_idx], y[test_idx]
        
        # Apply ADAPTIVE SMOTE-MRS
        engine = AdaptiveSMOTEMRS(R=5, ir_threshold=1.5)
        X_res, y_res = engine.fit_resample(X_tr, y_tr)
        
        # Train LightGBM
        lgb = LGBMClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            is_unbalance=True,  # Handle imbalance
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
        lgb.fit(X_res, y_res)
        
        preds = lgb.predict(X_te)
        probs = lgb.predict_proba(X_te)[:, 1]
        
        m['acc'].append(accuracy_score(y_te, preds))
        m['rec'].append(recall_score(y_te, preds))
        m['f1'].append(f1_score(y_te, preds))
        m['auc'].append(roc_auc_score(y_te, probs))
        
        print(f"  Fold {fold}: Recall={m['rec'][-1]:.4f}, Accuracy={m['acc'][-1]:.4f}")
    
    results.append({
        'Dataset': dataset_name,
        'Model': 'LightGBM',
        'Accuracy': round(np.mean(m['acc']), 4),
        'Recall': round(np.mean(m['rec']), 4),
        'F1': round(np.mean(m['f1']), 4),
        'AUC': round(np.mean(m['auc']), 4)
    })
    print(f"✅ {dataset_name} - LightGBM (Adaptive) Done")
    
    return results

# ========================================================
# 5. RUN ON BOTH DATASETS
# ========================================================
print("🚀 LightGBM with ADAPTIVE SMOTE-MRS - Pima & CKD\n")

all_results = []

# Dataset 1: Pima Diabetes
X1, y1, name1 = load_diabetes()
print(f"📊 {name1}: {len(y1)} samples, Class distribution: {np.bincount(y1)}")
all_results.extend(evaluate_lightgbm_adaptive(X1, y1, name1))

# Dataset 2: UCI CKD
X2, y2, name2 = load_ckd()
print(f"\n📊 {name2}: {len(y2)} samples, Class distribution: {np.bincount(y2)}")
all_results.extend(evaluate_lightgbm_adaptive(X2, y2, name2))

# Final Results Table
results_df = pd.DataFrame(all_results)
print("\n" + "="*70)
print("    LightGBM with ADAPTIVE SMOTE-MRS - BOTH DATASETS")
print("         (IR THRESHOLD = 1.5)")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)


# sTROKE BASE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ========================================================
# 1. CUSTOM SMOTE IMPLEMENTATION
# ========================================================
def custom_resample(X, y, k_neighbors=3):
    X = np.array(X)
    y = np.array(y)
    counts = pd.Series(y).value_counts()
    minority_class = counts.idxmin()
    X_min = X[y == minority_class]
    n_min = len(X_min)
    
    if n_min > 1:
        nn = NearestNeighbors(n_neighbors=min(n_min, k_neighbors + 1)).fit(X_min)
        dist, idxs = nn.kneighbors(X_min)
        
        synthetic = []
        for _ in range(counts.max() - n_min):
            i = random.randint(0, n_min - 1)
            neighbor = random.choice(idxs[i][1:])
            diff = X_min[neighbor] - X_min[i]
            synthetic.append(X_min[i] + random.random() * diff)
        
        if synthetic:
            X = np.vstack([X, np.array(synthetic)])
            y = np.append(y, [minority_class] * len(synthetic))
    
    return X, y

# ========================================================
# 2. BASE SMOTE-MRS IMPLEMENTATION
# ========================================================
class SMOTEMRS_Base:
    def __init__(self, R=5):
        self.R = R
        
    def fit_resample(self, X, y):
        kmeans = KMeans(n_clusters=self.R, random_state=42, n_init='auto')
        clusters = kmeans.fit_predict(X)
        
        X_res, y_res = [], []
        for r in range(self.R):
            mask = (clusters == r)
            if not any(mask): continue
            
            X_sub, y_sub = X[mask], y[mask]
            if len(np.unique(y_sub)) > 1:
                X_s, y_s = custom_resample(X_sub, y_sub)
                X_res.append(X_s)
                y_res.append(y_s)
            else:
                X_res.append(X_sub)
                y_res.append(y_sub)
                
        return np.vstack(X_res), np.concatenate(y_res)

# ========================================================
# 3. LOAD STROKE DATASET
# ========================================================
def load_stroke():
    df = pd.read_csv('/kaggle/input/datasets/fedesoriano/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv')
    df = df.drop('id', axis=1)
    df['bmi'].fillna(df['bmi'].median(), inplace=True)
    
    le = LabelEncoder()
    for col in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']:
        df[col] = le.fit_transform(df[col].astype(str))
    
    X = StandardScaler().fit_transform(df.drop('stroke', axis=1))
    y = df['stroke'].values
    return X, y

# ========================================================
# 4. LIGHTGBM EVALUATION
# ========================================================
def evaluate_lightgbm(X, y):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    print(f"\n📊 Stroke Prediction Dataset")
    
    m = {'acc': [], 'rec': [], 'f1': [], 'auc': []}
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[test_idx], y[test_idx]
        
        # Apply Base SMOTE-MRS
        engine = SMOTEMRS_Base(R=5)
        X_res, y_res = engine.fit_resample(X_tr, y_tr)
        
        # Train LightGBM
        lgb = LGBMClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            is_unbalance=True,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
        lgb.fit(X_res, y_res)
        
        preds = lgb.predict(X_te)
        probs = lgb.predict_proba(X_te)[:, 1]
        
        m['acc'].append(accuracy_score(y_te, preds))
        m['rec'].append(recall_score(y_te, preds))
        m['f1'].append(f1_score(y_te, preds))
        m['auc'].append(roc_auc_score(y_te, probs))
        
        print(f"  Fold {fold}: Recall={m['rec'][-1]:.4f}, Accuracy={m['acc'][-1]:.4f}")
    
    results.append({
        'Model': 'LightGBM',
        'Accuracy': round(np.mean(m['acc']), 4),
        'Recall': round(np.mean(m['rec']), 4),
        'F1': round(np.mean(m['f1']), 4),
        'AUC': round(np.mean(m['auc']), 4)
    })
    print(f"✅ Stroke Prediction - LightGBM (Base) Done")
    
    return pd.DataFrame(results)

# ========================================================
# 5. RUN
# ========================================================
print("🚀 LightGBM with Base SMOTE-MRS - Stroke Prediction Dataset\n")

X, y = load_stroke()
print(f"📊 Samples: {len(y)}")
print(f"📊 Class distribution: {np.bincount(y)}")
print(f"📊 Imbalance Ratio: {np.bincount(y)[0] / np.bincount(y)[1]:.2f}:1\n")

results = evaluate_lightgbm(X, y)

print("\n" + "="*60)
print("   LightGBM with BASE SMOTE-MRS - STROKE PREDICTION")
print("="*60)
print(results.to_string(index=False))
print("="*60)


# sTROKE ADAPTIVE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ========================================================
# 1. CUSTOM SMOTE IMPLEMENTATION
# ========================================================
def custom_resample(X, y, k_neighbors=3):
    X = np.array(X)
    y = np.array(y)
    counts = pd.Series(y).value_counts()
    minority_class = counts.idxmin()
    X_min = X[y == minority_class]
    n_min = len(X_min)
    
    if n_min > 1:
        nn = NearestNeighbors(n_neighbors=min(n_min, k_neighbors + 1)).fit(X_min)
        dist, idxs = nn.kneighbors(X_min)
        
        synthetic = []
        for _ in range(counts.max() - n_min):
            i = random.randint(0, n_min - 1)
            neighbor = random.choice(idxs[i][1:])
            diff = X_min[neighbor] - X_min[i]
            synthetic.append(X_min[i] + random.random() * diff)
        
        if synthetic:
            X = np.vstack([X, np.array(synthetic)])
            y = np.append(y, [minority_class] * len(synthetic))
    
    return X, y

# ========================================================
# 2. ADAPTIVE SMOTE-MRS IMPLEMENTATION ⭐
# ========================================================
class AdaptiveSMOTEMRS:
    def __init__(self, R=5, ir_threshold=1.5):
        self.R = R
        self.ir_threshold = ir_threshold
        
    def fit_resample(self, X, y):
        kmeans = KMeans(n_clusters=self.R, random_state=42, n_init='auto')
        clusters = kmeans.fit_predict(X)
        
        X_res, y_res = [], []
        
        for r in range(self.R):
            mask = (clusters == r)
            if not any(mask): 
                continue
            
            X_sub, y_sub = X[mask], y[mask]
            unique_classes = np.unique(y_sub)
            
            if len(unique_classes) < 2:
                X_res.append(X_sub)
                y_res.append(y_sub)
                continue
            
            counts = pd.Series(y_sub).value_counts()
            majority_count = counts.max()
            minority_count = counts.min()
            ir = majority_count / minority_count
            
            if ir >= self.ir_threshold:
                X_s, y_s = custom_resample(X_sub, y_sub)
                X_res.append(X_s)
                y_res.append(y_s)
            else:
                X_res.append(X_sub)
                y_res.append(y_sub)
                
        return np.vstack(X_res), np.concatenate(y_res)

# ========================================================
# 3. LOAD STROKE DATASET
# ========================================================
def load_stroke():
    df = pd.read_csv('/kaggle/input/datasets/fedesoriano/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv')
    df = df.drop('id', axis=1)
    df['bmi'].fillna(df['bmi'].median(), inplace=True)
    
    le = LabelEncoder()
    for col in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']:
        df[col] = le.fit_transform(df[col].astype(str))
    
    X = StandardScaler().fit_transform(df.drop('stroke', axis=1))
    y = df['stroke'].values
    return X, y

# ========================================================
# 4. LIGHTGBM EVALUATION WITH ADAPTIVE
# ========================================================
def evaluate_lightgbm_adaptive(X, y):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    print(f"\n📊 Stroke Prediction Dataset")
    
    m = {'acc': [], 'rec': [], 'f1': [], 'auc': []}
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[test_idx], y[test_idx]
        
        # Apply ADAPTIVE SMOTE-MRS
        engine = AdaptiveSMOTEMRS(R=5, ir_threshold=1.5)
        X_res, y_res = engine.fit_resample(X_tr, y_tr)
        
        # Train LightGBM
        lgb = LGBMClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            is_unbalance=True,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
        lgb.fit(X_res, y_res)
        
        preds = lgb.predict(X_te)
        probs = lgb.predict_proba(X_te)[:, 1]
        
        m['acc'].append(accuracy_score(y_te, preds))
        m['rec'].append(recall_score(y_te, preds))
        m['f1'].append(f1_score(y_te, preds))
        m['auc'].append(roc_auc_score(y_te, probs))
        
        print(f"  Fold {fold}: Recall={m['rec'][-1]:.4f}, Accuracy={m['acc'][-1]:.4f}")
    
    results.append({
        'Model': 'LightGBM',
        'Accuracy': round(np.mean(m['acc']), 4),
        'Recall': round(np.mean(m['rec']), 4),
        'F1': round(np.mean(m['f1']), 4),
        'AUC': round(np.mean(m['auc']), 4)
    })
    print(f"✅ Stroke Prediction - LightGBM (Adaptive) Done")
    
    return pd.DataFrame(results)

# ========================================================
# 5. RUN
# ========================================================
print("🚀 LightGBM with ADAPTIVE SMOTE-MRS - Stroke Prediction Dataset\n")

X, y = load_stroke()
print(f"📊 Samples: {len(y)}")
print(f"📊 Class distribution: {np.bincount(y)}")
print(f"📊 Imbalance Ratio: {np.bincount(y)[0] / np.bincount(y)[1]:.2f}:1\n")

results = evaluate_lightgbm_adaptive(X, y)

print("\n" + "="*60)
print(" LightGBM with ADAPTIVE SMOTE-MRS - STROKE PREDICTION")
print("     (IR THRESHOLD = 1.5)")
print("="*60)
print(results.to_string(index=False))
print("="*60)
